# Analyse Avancée de la Modularité : Bruit Aléatoire, Shuffle-Tests et Poids

Ce notebook implémente la partie pratique (**🔬 Tool**) sur la modularité des réseaux complexes :
1. **Q3 : Test de permutation (Shuffle-test)** sur le réseau des philosophes (*degree-preserving* et Erdős-Rényi).
2. **Q4 : Limites de la modularité sur le bruit pur** ($G(n, p)$ à taille et degré variables, extraction de GCC et instabilité stochastique).
3. **Q5 : Le club de karaté de Zachary et l'impact des poids** (séparation réelle vs Louvain, $Q$ pondéré vs non pondéré, score NMI).



In [ ]:
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics import normalized_mutual_info_score
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Modules prêts !')



## 0. Chargement de la composante géante (GCC) des philosophes


In [ ]:
# Chargement du réseau
if os.path.exists('philosophers.edgelist'):
    G = nx.read_weighted_edgelist('philosophers.edgelist')
else:
    df_edges = pd.read_csv('week4_philosophers_edges.tsv', sep='\t', comment='#')
    G = nx.Graph()
    for _, row in df_edges.iterrows():
        u, v, w = row['source'], row['target'], float(row['weight'])
        if G.has_edge(u, v):
            G[u][v]['weight'] += w
        else:
            G.add_edge(u, v, weight=w)

# Extraction de la GCC
gcc_nodes = max(nx.connected_components(G), key=len)
gcc = G.subgraph(gcc_nodes).copy()
print(f"GCC des philosophes : {gcc.number_of_nodes()} nœuds, {gcc.number_of_edges()} arêtes.")



## Tâche 1 (Q3) : Test de permutation (Shuffle-test) sur les philosophes
- 20 réseaux aléatoires préservant les degrés via `double_edge_swap`.
- 20 réseaux $G(n, m)$ d'Erdős-Rényi avec les mêmes $n$ et $m$.
- Calcul de la moyenne, écart-type et z-score du $Q$ réel.



In [ ]:
n = gcc.number_of_nodes()
m = gcc.number_of_edges()
base_seed = 42

# 1. Q réel avec Louvain
real_comms = nx.community.louvain_communities(gcc, seed=base_seed)
q_real = nx.community.modularity(gcc, real_comms)
print(f"Modularité réelle (Louvain) : Q_real = {q_real:.4f} ({len(real_comms)} communautés)")

# 2. 20 Degree-preserving shuffles
print("Génération des 20 shuffles préservant les degrés...")
q_shuffled = []
n_swaps = 3 * m
for i in range(20):
    g_rand = gcc.copy()
    nx.double_edge_swap(g_rand, nswap=n_swaps, max_tries=n_swaps * 5, seed=base_seed + i)
    c_s = nx.community.louvain_communities(g_rand, seed=base_seed)
    q_shuffled.append(nx.community.modularity(g_rand, c_s))

mean_shuff = np.mean(q_shuffled)
std_shuff = np.std(q_shuffled, ddof=1)
z_shuff = (q_real - mean_shuff) / std_shuff

print(f"Degree-preserving : μ = {mean_shuff:.4f}, σ = {std_shuff:.4f} --> z-score = {z_shuff:.2f}")

# 3. 20 Erdős-Rényi G(n, m)
print("Génération des 20 réseaux G(n, m)...")
q_er = []
for i in range(20):
    g_er = nx.gnm_random_graph(n, m, seed=base_seed + i)
    c_e = nx.community.louvain_communities(g_er, seed=base_seed)
    q_er.append(nx.community.modularity(g_er, c_e))

mean_er = np.mean(q_er)
std_er = np.std(q_er, ddof=1)
z_er = (q_real - mean_er) / std_er

print(f"Erdős-Rényi G(n, m) : μ = {mean_er:.4f}, σ = {std_er:.4f} --> z-score = {z_er:.2f}")

# Visualisation des distributions
plt.figure(figsize=(9, 4.5))
sns.kdeplot(q_shuffled, fill=True, color='royalblue', label=f'Degree-preserving (μ={mean_shuff:.3f}, z={z_shuff:.1f})')
sns.kdeplot(q_er, fill=True, color='forestgreen', label=f'Erdős-Rényi G(n, m) (μ={mean_er:.3f}, z={z_er:.1f})')
plt.axvline(q_real, color='crimson', linestyle='--', linewidth=2.5, label=f'Réseau réel (Q={q_real:.3f})')
plt.title('Test de Permutation (Q3) : Modularité Réelle vs Modèles Nuls', fontsize=13)
plt.xlabel('Modularité Q')
plt.ylabel('Densité')
plt.legend()
plt.tight_layout()
plt.show()



## Tâche 2 (Q4) : Limites de la modularité sur le bruit pur
1. $G(n, p)$ avec $n = 1000$ et $\langle k \rangle \in [1, 50]$ (tracé de $Q$ vs $\langle k \rangle$).
2. $G(n, p)$ avec $\langle k \rangle = 3$ et $n \in [100, 10\,000]$ (tracé de $Q$ vs $n$).
3. Recherche du réseau avec $Q \approx 0.51$, test de deux seeds et calcul de la NMI.



In [ ]:
# 1. Q vs k (n = 1000 fixe)
n_fixed = 1000
k_vals = [1, 2, 3, 4, 4.2, 5, 7, 10, 15, 20, 25, 30, 40, 50]
q_vs_k = []
stored_nets = []

for k in k_vals:
    p = k / (n_fixed - 1)
    G_rand = nx.erdos_renyi_graph(n_fixed, p, seed=base_seed)
    gcc_rand = G_rand.subgraph(max(nx.connected_components(G_rand), key=len)).copy()
    c = nx.community.louvain_communities(gcc_rand, seed=base_seed)
    q = nx.community.modularity(gcc_rand, c)
    q_vs_k.append(q)
    stored_nets.append((k, gcc_rand, q))

# 2. Q vs n (k = 3 fixe)
k_fixed = 3.0
n_vals = [100, 200, 500, 1000, 2000, 4000, 7000, 10000]
q_vs_n = []

for n_curr in n_vals:
    p = k_fixed / (n_curr - 1)
    G_rand = nx.erdos_renyi_graph(n_curr, p, seed=base_seed)
    gcc_rand = G_rand.subgraph(max(nx.connected_components(G_rand), key=len)).copy()
    c = nx.community.louvain_communities(gcc_rand, seed=base_seed)
    q_vs_n.append(nx.community.modularity(gcc_rand, c))

# Tracé des courbes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(k_vals, q_vs_k, marker='o', color='#1f77b4', linewidth=2)
ax1.axhline(0.51, color='crimson', linestyle=':', label='Seuil Q ≈ 0.51')
ax1.set_title('Modularité Q vs Degré Moyen $\langle k \rangle$ (n = 1000)', fontsize=12)
ax1.set_xlabel('$\langle k \rangle$')
ax1.set_ylabel('Modularité maximale Q')
ax1.legend()

ax2.plot(n_vals, q_vs_n, marker='s', color='#ff7f0e', linewidth=2)
ax2.set_xscale('log')
ax2.set_title('Modularité Q vs Taille n ($\langle k \rangle = 3$ fixe)', fontsize=12)
ax2.set_xlabel('Nombre de nœuds n (échelle log)')
ax2.set_ylabel('Modularité maximale Q')

plt.tight_layout()
plt.show()

# 3. Réseau avec Q le plus proche de 0.51
best_k, best_gcc, best_q = min(stored_nets, key=lambda x: abs(x[2] - 0.51))
print(f"Réseau sélectionné : k={best_k}, {best_gcc.number_of_nodes()} nœuds GCC, Q = {best_q:.4f}")

# Deux exécutions de Louvain
c1 = nx.community.louvain_communities(best_gcc, seed=1)
c2 = nx.community.louvain_communities(best_gcc, seed=2)

nodes_list = sorted(list(best_gcc.nodes()))
n2c1 = {node: i for i, comm in enumerate(c1) for node in comm}
n2c2 = {node: i for i, comm in enumerate(c2) for node in comm}

nmi_noise = normalized_mutual_info_score([n2c1[u] for u in nodes_list], [n2c2[u] for u in nodes_list])

print(f"Louvain Seed 1 : {len(c1)} communautés, Q = {nx.community.modularity(best_gcc, c1):.4f}")
print(f"Louvain Seed 2 : {len(c2)} communautés, Q = {nx.community.modularity(best_gcc, c2):.4f}")
print(f"NMI entre Seed 1 et Seed 2 sur bruit pur : {nmi_noise:.4f}")



## Tâche 3 (Q5) : Le club de karaté de Zachary et l'effet des poids
1. Extraction de la séparation réelle (`Mr. Hi` vs `Officer`).
2. Calcul de $Q$ de la séparation réelle AVEC et SANS les poids.
3. Exécution de Louvain sans les poids (`weight=None`) et calcul de la NMI.



In [ ]:
G_karate = nx.karate_club_graph()

# 1. Séparation réelle
mr_hi = {n for n, d in G_karate.nodes(data=True) if d.get('club') == 'Mr. Hi'}
officer = {n for n, d in G_karate.nodes(data=True) if d.get('club') == 'Officer'}
ground_truth = [mr_hi, officer]

# 2. Modularité avec et sans poids
q_truth_weighted = nx.community.modularity(G_karate, ground_truth, weight='weight')
q_truth_unweighted = nx.community.modularity(G_karate, ground_truth, weight=None)

print(f"Séparation réelle :")
print(f" - Avec poids (weight='weight') : Q = {q_truth_weighted:.4f}")
print(f" - Sans poids (weight=None)     : Q = {q_truth_unweighted:.4f}")
print(f" - Différence ΔQ                : {q_truth_weighted - q_truth_unweighted:+.4f}")

# 3. Louvain sans poids
louvain_unw = nx.community.louvain_communities(G_karate, weight=None, seed=42)
q_louvain_unw = nx.community.modularity(G_karate, louvain_unw, weight=None)

nodes_k = sorted(list(G_karate.nodes()))
truth_labels = [0 if G_karate.nodes[u]['club'] == 'Mr. Hi' else 1 for u in nodes_k]
n2c_l = {node: i for i, comm in enumerate(louvain_unw) for node in comm}
louvain_labels = [n2c_l[u] for u in nodes_k]

nmi_karate = normalized_mutual_info_score(truth_labels, louvain_labels)

print(f"\nLouvain non pondéré (weight=None) :")
print(f" - Nombre de communautés : {len(louvain_unw)}")
print(f" - Modularité Q          : {q_louvain_unw:.4f}")
print(f" - NMI avec vérité terrain: {nmi_karate:.4f}")

